# Building conversational applications with CrewAI

In this notebook, you learn how to use CrewAI to create collaborative AI agents that can work together to solve complex tasks.

This demonstrates the same functionality as the Strands Agents example, but using CrewAI's multi-agent approach.

## Environment setup

In this task, you set up your environment and install the required packages for CrewAI.

In [ ]:
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

model_id = "amazon.nova-lite-v1:0"

## Define Tools for CrewAI

In this task, we define the tools that our CrewAI agents will use.

In [ ]:
@tool
def calculator_tool(oper1: int, oper2: int, operator: str) -> str:
    """
    Perform basic arithmetic operations on two numbers.
    
    Args:
        oper1: First operand
        oper2: Second operand  
        operator: Operation to perform (+, -, *, / or add, subtract, multiply, divide)
    
    Returns:
        Result of the calculation as a string
    """
    def add(a, b):
        return a + b

    def subtract(a, b):
        return a - b

    def multiply(a, b):
        return a * b

    def divide(a, b):
        if b == 0:
            return "Cannot divide by zero"
        else:
            return float(a / b)
    
    if operator == '+' or operator == 'add':
        return str(add(oper1, oper2))
    elif operator == '-' or operator == 'subtract':
        return str(subtract(oper1, oper2))
    elif operator == '*' or operator == 'multiply':
        return str(multiply(float(oper1), float(oper2)))
    elif operator == '/' or operator == 'divide':
        return str(divide(oper1, oper2))
    else:
        return "Invalid operator"

In [ ]:
@tool
def search_tool(query: str) -> str:
    """
    Search with DuckDuckGo for information about a topic.
    
    Args:
        query: The search query
    
    Returns:
        DuckDuckGo search results as a string
    """
    tool = DuckDuckGoSearchRun()
    result = tool.invoke(query)
    return result

## Create CrewAI Agents and Tasks

Now we create specialized agents for different tasks and define how they work together.

In [ ]:
# Create the Bedrock LLM
llm = LLM(
    model="bedrock/us.amazon.nova-lite-v1:0"
)

# Create agent
agent = Agent(
    role="Expert in searching and calculations",
    goal="Use available tools to find Find accurate information on any topic using tools",
    backstory="You are a helpful assistant that can perform calculations and search for information on DuckDuckGo. Use the available tools when needed to provide accurate and helpful responses.",
    tools=[search_tool, calculator_tool],
    llm=llm,
    verbose=True
)

## Test the CrewAI System

Now let's test our CrewAI system with the same queries used in the Strands example.

In [ ]:
def run_crew_task(user_query: str):
    """Run a CrewAI task based on user query"""
    
    # Create task
    task = Task(
        description=f"Answer this query: {user_query}",
        expected_output="A comprehensive answer to the user's query",
        agent=agent
    )
    
    # Create crew
    crew = Crew(
        agents=[agent],
        tasks=[task],
        verbose=True
    )
    
    # Execute
    result = crew.kickoff()
    return result

In [ ]:
# Test with the same complex query from Strands example
query = "What is Amazon SageMaker? What is launch year multiplied by 2"

print(f"Query: {query}\n")
print("CrewAI Response:")
print("=" * 50)

response = run_crew_task(query)
print(response)

In [ ]:
# Test with calculation-only query
calc_query = "What is 156 multiplied by 23?"
print(f"Query: {calc_query}")
print(f"Response: {run_crew_task(calc_query)}")
print("\n" + "="*50 + "\n")

In [ ]:
# Test with search-only query
search_query = "Tell me about artificial intelligence"
print(f"Query: {search_query}")
print(f"Response: {run_crew_task(search_query)}")
print("\n" + "="*50 + "\n")